---
title: Week 5 Tutorial 1, Orchestra Simulation
subtitle: Orchestra Scenario, Fluorite & Gypsum simulation
author:
  - name: Timo Heimovaara
    affiliations: Delft University of Technology, department of Geoscience & Engineering
    orcid: 
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-01-07
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Orchestra as a tool
This example illustrates how to use Python and ORCHESTRA to simulate the Ca - F - Gypsum example from Appelo and Postma chapter 4.
The next python cells imports the necessary libararies and checks the path to the input files required for the Orchestra simulation. Please note that this is only required for the jupyter-book version. For a stand-alone version of this notebook you need to start jupyter-lab from the directory with the input files.

In [1]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

%matplotlib widget
sns.set()

orchestra_path = "." 
# print(orchestra_path)

## Define chemical system: Calcite, Fluorite, Gypsum, Rhodochrosite water using the ORCHESTRA-GUI
In order to solve a chemical equilibrium problem with pyOrchestra, we need to define our chemical system first. Orchestra defines this system with a number of text files. The most important one is the so-called chemistry input file (*chemistry1.inp*). This file is most easily made using the Orchestra GUI which can be accessed by clicking on the orchestrat2023.jar file.

- Calcite is $\text{CaCO}_3\text(s)$
- Fluorite is $\text{CaF}_2\text(s)$
- Gypsum is $\text{CaSO}_4:2\text{H}_2\text{O (s)}$
- Rhodochrosite is $\text{MnaCO}_3\text(s)$

The master species table looks like

|Primary entity|Phase| Input Variable|Fix log activity|Log activity|Concentration|Phase|Expression|
|--|--|--|--|--|--|--|--|
|Ar[g]|gas||x|-15.0||||
|CO3-2|diss ||||12.65|tot||||
|Ca+2 |diss ||||15.0|tot||
|Fe-  |diss   ||||5.0|tot||
|H+   |diss|pH|x|-7.0|||H.logact=-pH|
|H2O  |liter||x|0.0||||
|Mn+2  |diss   ||||5.0|tot||
|Na+   |diss   ||||0.1|tot||
|SO4-2 |diss   ||||5.0|tot||


Orchestra will use these primary entities, and their initial values to calculate the total elemental composition in the complete system. The fixed log-activties indicate that the total can vary during the simulation, as long as the log-activity condition is maintained. We assume that Orchestra uses the pH to maintain the chargebalance. 

### Phases & Reactions
On the **Phases & Reactions** tab shown in figure [](#phases_reactions) we define the reaction network of our system. The schematic on the left-hand side is used to define how the different phases (gas, liquid and solids) are connected, which we will ignore for now.

The table on this tab shows all possible reactions in this system. You should recognize that this table originates from the log-transformed mass action law. The table shows the logK-value, in which phase the species can be found, followed by the stoichiometry of the reaction as a function of the master-species or convenient secondary species in the system. 

For this example we need to select Calcite[s], Fluorite[s], Gypsum[s], Rhodochrosite[], and CO2[g] because we want these minerals to precipitate or to exchange with the gas phase, Orchestra will force the SI to be zero if the Ion activity product is larger that the logK value.

### Other tabs
On the **Activity correction** tab we select the model equation used by ORCHESTRA to calculate the activities of the species in solution. We will use the Davies model and mark the field **Calculate Ionic Strenghth** which you can check for your self. 
On the **Settings** tab we mark the field **Include SI minerals**. This last item allows us to evaluate which other minerals in the system might become oversaturated, we then may choose to include these in the precipitation reactions as well.

The other tabs we leave as is, but you should take a look at these tabs anyway. On the **Variables** tab the user can set all kinds of constants for the calculations. However, many of these will be "overruled" by us in the Python simulations. The **Adsorption models** tab is used to calculate adsortion of ionic compounds to all kinds of surfaces, lies beyond the scope of this course. The **Predominance Diagram** tab will be used in a later lecture when we discuss redox reactions. Finally the **Output selector** tab is useful to find out what variables are present in our current ORCHESTRA simulation.

### Using Orchestra to run the scenario
It is possible to use the ORCHESTRA GUI to run the simulation, and this has been prepared using the **Input** and **Output** tabs on the right of the window. Please have look at these tabs and they should be rather "self explanatory". 



## Running the scenarios in a Python notebook
We will use a Python notebook and Orchestra to calculate:
1. the initial state, allowing us to calculate the water composition in the geothermal aquifer
2. the final state in the bottle allowing us to calculate the final composition of the water in the bottle and the amount and type of minerals that have precipitated while conditions changed in the bottle.

In order to run the in a notebook we have to follow a systematic approach which consists of the following standard steps:
1. define the domain for our problem
2. define the primary species which change during our scenario
3. initialize the problem
4. run the problem
5. process the output

### 1. Define the domain
We assume a simulation in 1 liter (or 1kg) of water. 
For the initial state the amount of gas volume is not really necessary so we set it to a very small number. In the final state the gas volume is relevant!

### 2. Define the primary species
The primary species are as defined above with the ORCHESTRA GUI. Please note that the values required by ORCHESTRA have to be in moles. In order to get moles/liter the volume of water has to be set to 1 liter, and the density of water need to be given as well.

### 3. Initialise the problem 
The initial condition for our problem is the situation described in the assignment.
The final state is a situation where the solution composition from the initial state is transferred in to the bottle and allowed to cool-down and degas.

Preliminary analysis on solid samples taken from the aquifer show that total amounts of the species are as shown in the table above. The CO3-2.tot was found by a combination of chemical analysis and measurement of the partial $\text{CO}_2\text{[g]}$ pressure in the aquifer. The logactivity for $\text{Ar[g]}$ is set to a small number in order to allow for an inert gas present in the bottle after filling. 


Once pyOrchestra is initialized, running a simulation consists of a series of steps where the values of the required set of input variables are passed via *InVARS* to ORCHESTRA, after which a set of corresponding output variables are passed back in *OutVars*.

ORCHESTRA is initialized in pyOrchestra using the inputfile *chemistry1.inp*, created above with the ORCHESTRA-GUI. After initialization in Python, we know which variables will be passed through *OutVars* and can be used in *InVars*.

The following code shows how to do this.


In [2]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry1.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    InVars = np.array([
        'Ca+2.tot', 'Mn+2.tot', 'Na+.tot',
        'F-.tot', 'SO4-2.tot',
        'CO3-2.tot', 'Ar[g].logact',
        'watervolume', 'gasvolume_fixed',
        'T'
        ])
    
    # We select the output from Orchestra we need to use
    OutVars = np.array([
        'pH', 'H+.con', 'OH-.con', 'H+.tot', 
        'CO3-2.tot', 'CO3-2.con', 'CO3-2.logact','CO3-2.diss',
        'HCO3-.tot', 'HCO3-.con', 'HCO3-.logact',
        'H2CO3.tot', 'H2CO3.con', 'H2CO3.logact',
        'CO2[g].tot', 'CO2[g].con', 'CO2[g].logact',
        'Ar[g].tot', 'Ar[g].con', 'Ar[g].logact',
        'F-.tot', 'F-.con', 'F-.logact','F-.diss',
        'Ca+2.tot', 'Ca+2.con', 'Ca+2.logact', 'Ca+2.diss', 
        'Mn+2.tot', 'Mn+2.con', 'Mn+2.logact', 'Mn+2.diss',
        'CaF+.con', 'CaHCO3+.con', 'CaOH+.con', 'CaSO4.con',
        'H2F2.con', 'HF.con', 'HF2-.con', 'H2SO4.con', 'HSO4-.con', 'SO4-2.con', 'SO4-2.diss',
        'MnF+.con', 'MnHCO3+.con', 'MnOH+.con', 'MnSO4.con', 'Mn[OH]3-.con',
        'Mn[OH]4-2.con', 'Na+.diss',
        'Calcite[s].si', 'Calcite[s].tot', 'Calcite[s].logact',
        'Fluorite[s].si', 'Fluorite[s].tot', 'Fluorite[s].logact',
        'Gypsum[s].si', 'Gypsum[s].tot', 'Gypsum[s].logact',
        'Rhodochrosite[s].si', 'Rhodochrosite[s].tot', 'Rhodochrosite[s].logact',
        'T', 'I', 'chargebalance', 'watervolume', 'gasvolume', 'pressure' ])

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    p = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    p.initialise(InputFile, NoCells, InVars, OutVars)


Reading and expanding calculator new stylechemistry1.inp
Scanning file: chemistry1.inp
Scanning file: objects2026_THe.txt
Scanning file: chemistry1.inp
Scanning file: objects2026_THe.txt
Including file: chemistry1.inp
Scanning file: objects2026_THe.txt
0.022 sec.
	Reading variables .... 0.012 s
testing:
13:Ca+2.tot
18:Mn+2.tot
20:Na+.tot
15:F-.tot
22:SO4-2.tot
11:CO3-2.tot
9:Ar[g].logact
27:watervolume
28:gasvolume_fixed
4:T
16:pH
29:H+.con
30:OH-.con
31:H+.tot
11:CO3-2.tot
32:CO3-2.con
10:CO3-2.logact
33:CO3-2.diss
34:HCO3-.tot
35:HCO3-.con
36:HCO3-.logact
37:H2CO3.tot
38:H2CO3.con
39:H2CO3.logact
40:CO2[g].tot
41:CO2[g].con
42:CO2[g].logact
43:Ar[g].tot
44:Ar[g].con
9:Ar[g].logact
15:F-.tot
45:F-.con
14:F-.logact
46:F-.diss
13:Ca+2.tot
47:Ca+2.con
12:Ca+2.logact
48:Ca+2.diss
18:Mn+2.tot
49:Mn+2.con
17:Mn+2.logact
50:Mn+2.diss
51:CaF+.con
52:CaHCO3+.con
53:CaOH+.con
54:CaSO4.con
55:H2F2.con
56:HF.con
57:HF2-.con
58:H2SO4.con
59:HSO4-.con
60:SO4-2.con
61:SO4-2.diss
62:MnF+.con
63:MnHCO

Please note that the output of this code is what ORCHESTRA echos back. ORCHESTRA uses a set of variables in order to store the input variables and the results of the calculations, in this case 33. The top part of the output shows the output requested by us through *OutVars* together with the values used during initialization.

### Step 1 Initial condition high temperature (75 degrees C) in equilibrium with rocks

We take the 'Ca+2.tot', 'Mn+2.tot','CO3-2.tot', 'F-.tot', 'SO4-2.tot' values from the table above.
Orchestra returns all relevant parameters describing the equilibrium state of our system. The total amounts of dissolved master species are given with the ".diss" extension. We print this output in order to know what the total dissolved amounts of the master species are in the bottle that is filled with groundwater.

In [3]:
# %%
# Step 1, initial condition.
# InVars need to contain floats
IN = np.array([np.ones_like(InVars)]).astype(float)

IN[0][np.where(InVars == 'Ca+2.tot')] = 15 #mol/liter
IN[0][np.where(InVars == 'Mn+2.tot')] = 5 #mol/liter
IN[0][np.where(InVars == 'CO3-2.tot')] = 12.65 #mol/liter
IN[0][np.where(InVars == 'Ar[g].logact')] = -15 #log10 (1 atm)
IN[0][np.where(InVars == 'F-.tot')] = 5 #mol/liter
IN[0][np.where(InVars == 'SO4-2.tot')] = 5 #mol/liter
IN[0][np.where(InVars == 'Na+.tot')] = 0.1 #mol/liter
IN[0][np.where(InVars == 'watervolume')] = 1.0 #
IN[0][np.where(InVars == 'gasvolume_fixed')] = 1e-15 #
IN[0][np.where(InVars == 'gas_type')] = 0 # gas_type = 1: fixed pressure, gas_type=0 fixed volume

IN[0][np.where(InVars== 'T')] = 273.15 + 75 # temperature in K


# Calculate equilibrium conditions for initial situation

OUT = p.set_and_calculate(IN)
Res_df = pd.DataFrame(OUT,columns=OutVars,index=['Initial'])
print(r"Step 1: Initial condition")

table_md1 = Res_df[['Ca+2.diss', 'Mn+2.diss', 'CO3-2.diss', 'SO4-2.diss', 'F-.diss','Na+.diss', 'Calcite[s].tot', 'Fluorite[s].tot', 'Gypsum[s].tot', 'Rhodochrosite[s].tot']].to_markdown()
table_md2 = Res_df[['pH','CO2[g].logact', 'CO2[g].con', 'Ar[g].logact', 'Ar[g].con', 'pressure', 'T']].to_markdown()

#--- Reporting / Output ---] 
display(Markdown(table_md1))
display(Markdown(table_md2))


Step 1: Initial condition


|         |   Ca+2.diss |   Mn+2.diss |   CO3-2.diss |   SO4-2.diss |     F-.diss |   Na+.diss |   Calcite[s].tot |   Fluorite[s].tot |   Gypsum[s].tot |   Rhodochrosite[s].tot |
|:--------|------------:|------------:|-------------:|-------------:|------------:|-----------:|-----------------:|------------------:|----------------:|-----------------------:|
| Initial |   0.0133462 | 0.000353065 |      0.11609 |      0.04717 | 0.000878206 |        0.1 |          7.53426 |           2.49956 |         4.95283 |                4.99965 |

|         |      pH |   CO2[g].logact |   CO2[g].con |   Ar[g].logact |   Ar[g].con |   pressure |      T |
|:--------|--------:|----------------:|-------------:|---------------:|------------:|-----------:|-------:|
| Initial | 5.56136 |        0.886281 |      7.69629 |            -15 |       1e-15 |    7.69629 | 348.15 |

### Final situation, water in a bottle with 25 ml gasvolume at 25 degrees C

In [4]:
# %%
# Step 2, move 1 liter of ground water in to a bottle with a gasvolume of 25 ml, filled with 1 atm Ar[g].
# InVars can be taken form Res_df.loc['Initial']



IN[0][np.where(InVars == 'Ca+2.tot')] = Res_df.loc['Initial','Ca+2.diss'] #mol/liter
IN[0][np.where(InVars == 'Mn+2.tot')] = Res_df.loc['Initial','Mn+2.diss'] #mol/liter
IN[0][np.where(InVars == 'CO3-2.tot')] = Res_df.loc['Initial','CO3-2.diss'] #log10 (1 atm)
IN[0][np.where(InVars == 'Ar[g].logact')] = 0 #log10 (1 atm)
IN[0][np.where(InVars == 'F-.tot')] = Res_df.loc['Initial','F-.diss'] #log10 (1 atm)
IN[0][np.where(InVars == 'SO4-2.tot')] = Res_df.loc['Initial','SO4-2.diss']	 #log10 (1 atm)
IN[0][np.where(InVars == 'Na+.tot')] = Res_df.loc['Initial','Na+.diss']	 #log10 (1 atm)
IN[0][np.where(InVars == 'watervolume')] = 1.0 #
IN[0][np.where(InVars == 'gasvolume_fixed')] = 25e-3 #

IN[0][np.where(InVars== 'T')] = 273.15 + 25 # temperature in K
#print(IN[0])
# Calculate equilibrium conditions for initial situation
OUT = p.set_and_calculate(IN)

Res_df = pd.concat([Res_df, pd.DataFrame(OUT,columns=OutVars,index=['Final'])], ignore_index=False)

table_md1 = Res_df[['Ca+2.diss', 'Mn+2.diss', 'CO3-2.diss', 'SO4-2.diss', 'F-.diss','Na+.diss', 'Calcite[s].tot', 'Fluorite[s].tot', 'Gypsum[s].tot', 'Rhodochrosite[s].tot']].to_markdown()
table_md2 = Res_df[['pH','CO2[g].logact', 'CO2[g].con', 'Ar[g].logact', 'Ar[g].con','pressure', 'T']].to_markdown()


#--- Reporting / Output ---] 
display(Markdown(table_md1))
display(Markdown(table_md2))




|         |   Ca+2.diss |   Mn+2.diss |   CO3-2.diss |   SO4-2.diss |     F-.diss |   Na+.diss |   Calcite[s].tot |   Fluorite[s].tot |   Gypsum[s].tot |   Rhodochrosite[s].tot |
|:--------|------------:|------------:|-------------:|-------------:|------------:|-----------:|-----------------:|------------------:|----------------:|-----------------------:|
| Initial |   0.0133462 | 0.000353065 |    0.11609   |     0.04717  | 0.000878206 |        0.1 |          7.53426 |       2.49956     |      4.95283    |            4.99965     |
| Final   |   0.0114199 | 0.000160669 |    0.0783238 |     0.045501 | 0.000363732 |        0.1 |          0       |       0.000257237 |      0.00166899 |            0.000192396 |

|         |      pH |   CO2[g].logact |   CO2[g].con |   Ar[g].logact |   Ar[g].con |   pressure |      T |
|:--------|--------:|----------------:|-------------:|---------------:|------------:|-----------:|-------:|
| Initial | 5.56136 |        0.886281 |      7.69629 |            -15 |       1e-15 |    7.69629 | 348.15 |
| Final   | 6.04172 |        0.13726  |      1.3717  |              0 |       1     |    2.3717  | 298.15 |

In [5]:
Res_df.loc['Initial','Ca+2.diss']


np.float32(0.01334617)

## Analysis
The solubility of Calcite and Siderite at 50 $^o$C increases with partial CO2 pressure. At 10 atm we see that the solubility of Calcite exceeds the 0.01 mol/liter of Ca+2 in solution. If the amount of Calcite would be higher, we the concentration of Ca+2 increases. Same story for Fe+2, but the solubility increases less quickly as Ca+2 from Calcite. 